# Experiment: 离散布尔马尔可夫系统中的 Main Complex

这个 notebook 做三件事：

- 构造一个 5 节点的离散布尔马尔可夫动力系统；
- 严格按 IIT 2.0 的思路，对所有候选子系统枚举划分并计算状态依赖的 $\Phi$，再按 $\Phi$ 从大到小输出 complex；
- 再用 EI 分解中的协同信息做一个平行对照，检查“如果用 EI 协同来选 complex，是否会与 IIT 2.0 一致”。

为了让结果更直观，图中的文字保持英文，notebook 的解释全部使用中文。


In [1]:
from __future__ import annotations

import sys
from itertools import combinations
from pathlib import Path

import numpy as np
from IPython.display import HTML, Markdown, display

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'utils.py').exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break

from utils import (
    discrete_integrated_information,
    discrete_subset_synergy,
    enumerate_binary_states,
    find_discrete_complexes,
    render_static_causal_graph_svg,
)



## 1. 构造一个 5 节点布尔网络

这里我们用一个“3 节点核心 + 2 节点外围 readout”的结构：

- `A, B, C` 构成强耦合核心；
- 三个核心节点都由另外两个核心节点的 `AND` 联合决定；
- `D, E` 不再反馈回核心，而是作为核心的前馈 readout，均由 `A_t AND B_t` 驱动。

这样设计的目的，是让网络既足够大，又能明显区分“真正整合成整体的核心”和“只是被核心驱动的外围节点”。


In [2]:
NODE_LABELS = ['A', 'B', 'C', 'D', 'E']
N_NODES = len(NODE_LABELS)
CURRENT_STATE = (0, 0, 0, 0, 0)


def next_state(state: tuple[int, ...]) -> tuple[int, ...]:
    a, b, c, d, e = (int(value) for value in state)
    return (
        int(b and c),
        int(a and c),
        int(a and b),
        int(a and b),
        int(a and b),
    )


def build_demo_tpm() -> np.ndarray:
    states = enumerate_binary_states(N_NODES)
    index_by_state = {tuple(state.tolist()): idx for idx, state in enumerate(states)}
    tpm = np.zeros((len(states), len(states)), dtype=float)
    for row_index, state in enumerate(states):
        future = next_state(tuple(int(bit) for bit in state))
        tpm[row_index, index_by_state[future]] = 1.0
    return tpm


SYSTEM_TPM = build_demo_tpm()
ALL_STATES = enumerate_binary_states(N_NODES)
CURRENT_STATE_INDEX = int(np.flatnonzero(np.all(ALL_STATES == np.asarray(CURRENT_STATE), axis=1))[0])

STATIC_GRAPH_SVG = render_static_causal_graph_svg(
    'Boolean causal topology',
    node_labels=NODE_LABELS,
    directed_edges=[],
    hyperedges=[
        {'sources': (1, 2), 'target': 0, 'label': 'AND'},
        {'sources': (0, 2), 'target': 1, 'label': 'AND'},
        {'sources': (0, 1), 'target': 2, 'label': 'AND'},
        {'sources': (0, 1), 'target': 3, 'label': 'AND'},
        {'sources': (0, 1), 'target': 4, 'label': 'AND'},
    ],
    subtitle='Core nodes A/B/C are recurrently coupled; D/E are downstream readouts.',
    width=720,
    height=360,
)

display(HTML(STATIC_GRAPH_SVG))
print('当前考察的系统状态:', CURRENT_STATE)
print('该状态是否为不动点:', next_state(CURRENT_STATE) == CURRENT_STATE)



当前考察的系统状态: (0, 0, 0, 0, 0)
该状态是否为不动点: True


## 2. 先看 IIT 2.0 的 $\Phi$ 计算流程

对每个候选子系统 $S$，我们都做下面几步：

1. 取出子系统到自身下一时刻的精确离散 TPM；
2. 计算当前子系统状态对应的 state-dependent effective information；
3. 枚举这个子系统的所有非平凡划分；
4. 找到最弱划分，也就是 MIP；
5. 用 MIP 上的 partitioned effective information 作为该子系统的 $\Phi$。

最后，对所有子系统按 $\Phi$ 排序，并根据包含关系筛选出 IIT 2.0 意义下的 complex / main complex。


In [3]:
def subset_label(indices: tuple[int, ...] | list[int]) -> str:
    return '{' + ', '.join(NODE_LABELS[index] for index in indices) + '}'


def partition_label(partition) -> str:
    if not partition:
        return '-'
    return ' | '.join(subset_label(tuple(block)) for block in partition)


phi_ranking = []
for size in range(1, N_NODES + 1):
    for subset in combinations(range(N_NODES), size):
        result = discrete_integrated_information(
            SYSTEM_TPM,
            n_nodes=N_NODES,
            subset_indices=subset,
            current_state=CURRENT_STATE,
        )
        phi_ranking.append(result)

phi_ranking.sort(
    key=lambda item: (-float(item['phi']), -len(item['subset_indices']), tuple(item['subset_indices']))
)
phi_complexes = find_discrete_complexes(
    SYSTEM_TPM,
    n_nodes=N_NODES,
    current_state=CURRENT_STATE,
)

print(f'总共枚举了 {len(phi_ranking)} 个候选子系统。')
print(f'其中 IIT 2.0 意义下的 complex 数量为 {len(phi_complexes)}。')
print('按 Phi 排序后的第一名:', subset_label(phi_ranking[0]['subset_indices']), f"Phi={phi_ranking[0]['phi']:.6f}")



总共枚举了 31 个候选子系统。
其中 IIT 2.0 意义下的 complex 数量为 3。
按 Phi 排序后的第一名: {A, B, C} Phi=1.000000


## 3. 按 $\Phi$ 从大到小输出 IIT 2.0 的 complex

下表只保留通过 IIT 2.0 complex 筛选的子系统，并按 $\Phi$ 从大到小排列。


In [4]:
def render_rank_table(rows, *, title: str, score_key: str, extra_cols: list[tuple[str, str]] | None = None) -> None:
    extra_cols = extra_cols or []
    headers = ['rank', 'subset', score_key, 'MIP'] + [label for label, _ in extra_cols]
    html_parts = [
        f"<h4 style='margin:10px 0 8px 0;'>{title}</h4>",
        "<table style='border-collapse:collapse;font-size:13px;'>",
        "<thead><tr>" + ''.join(
            f"<th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>{header}</th>"
            for header in headers
        ) + "</tr></thead><tbody>",
    ]
    for rank, row in enumerate(rows, start=1):
        values = [
            str(rank),
            subset_label(row['subset_indices']),
            f"{row[score_key]:.6f}",
            partition_label(row.get('mip_partition', ())),
        ]
        for _, key in extra_cols:
            value = row[key]
            if isinstance(value, float):
                values.append(f"{value:.6f}")
            else:
                values.append(str(value))
        html_parts.append(
            '<tr>' + ''.join(
                f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{value}</td>"
                for value in values
            ) + '</tr>'
        )
    html_parts.append('</tbody></table>')
    display(HTML(''.join(html_parts)))


phi_display_rows = []
for row in phi_complexes:
    phi_display_rows.append({
        'subset_indices': row['subset_indices'],
        'phi': float(row['phi']),
        'mip_partition': row['mip_partition'],
        'main_complex': 'yes' if row['is_main_complex'] else 'no',
    })

render_rank_table(
    phi_display_rows,
    title='IIT 2.0 complexes sorted by Phi',
    score_key='phi',
    extra_cols=[('main complex', 'main_complex')],
)



rank,subset,phi,MIP,main complex
1,"{A, B, C}",1.000000,{A} | {B} | {C},yes
2,"{A, B, D}",0.500000,{A} | {B} | {D},yes
3,"{A, B, E}",0.500000,{A} | {B} | {E},yes


## 4. 详细查看排名第一的子系统

下面把第一名子系统的所有划分都列出来。这样可以直接看到它的 MIP 是谁，以及为什么它的 $\Phi$ 最高。


In [5]:
leader = phi_complexes[0]
leader_partition_rows = sorted(
    leader['partition_results'],
    key=lambda item: (item['normalized_partition_ei'], item['partition_ei']),
)

summary_lines = [
    f"- 排名第一的 complex: {subset_label(leader['subset_indices'])}",
    f"- 当前子系统状态: {leader['subset_state']}",
    f"- Whole EI: {leader['whole_ei']:.6f}",
    f"- Phi: {leader['phi']:.6f}",
    f"- MIP: {partition_label(leader['mip_partition'])}",
]
display(Markdown('\n'.join(summary_lines)))

html_parts = [
    "<table style='border-collapse:collapse;font-size:13px;'>",
    "<thead><tr><th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>partition</th>"
    "<th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>partition EI</th>"
    "<th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>normalized EI</th></tr></thead><tbody>",
]
for row in leader_partition_rows:
    html_parts.append(
        '<tr>'
        f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{partition_label(row['partition'])}</td>"
        f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{row['partition_ei']:.6f}</td>"
        f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{row['normalized_partition_ei']:.6f}</td>"
        '</tr>'
    )
html_parts.append('</tbody></table>')
display(HTML(''.join(html_parts)))



- 排名第一的 complex: {A, B, C}
- 当前子系统状态: (0, 0, 0)
- Whole EI: 1.000000
- Phi: 1.000000
- MIP: {A} | {B} | {C}

partition,partition EI,normalized EI
{A} | {B} | {C},1.000000,0.500000
"{A} | {B, C}",0.821928,0.821928
"{C} | {A, B}",0.821928,0.821928
"{B} | {A, C}",0.821928,0.821928


## 5. 用 EI 分解的协同信息做一个平行比较

现在换一个思路。对每个候选子系统 $S$，我们不再用 MIP 定义 $\Phi$，而是直接计算一个源侧 EI 协同分数：

$$
\mathrm{Syn}_{\mathrm{EI}}(S) = EI(S_t 	o S_{t+1}) - \sum_{i \in S} EI(i_t 	o S_{t+1}).
$$

这相当于把子系统的 source-side finest partition 固定为单节点划分，然后看“整体 EI 超过单节点和”的那一部分联合增益。

注意：这不是 IIT 2.0 的定义，只是一个有可比性的替代选择标准。


In [6]:
synergy_ranking = []
for size in range(1, N_NODES + 1):
    for subset in combinations(range(N_NODES), size):
        score = discrete_subset_synergy(
            SYSTEM_TPM,
            n_nodes=N_NODES,
            subset_indices=subset,
        )
        synergy_ranking.append(score)

synergy_ranking.sort(
    key=lambda item: (-float(item['synergy']), -len(item['subset_indices']), tuple(item['subset_indices']))
)


def select_synergy_complexes(rows):
    score_by_subset = {tuple(row['subset_indices']): float(row['synergy']) for row in rows}
    selected = []
    for row in rows:
        subset = tuple(row['subset_indices'])
        score = float(row['synergy'])
        if score <= 0.0:
            continue
        if any(
            set(subset).issubset(other_subset) and set(subset) != set(other_subset) and other_score > score
            for other_subset, other_score in score_by_subset.items()
        ):
            continue
        is_main = all(
            other_score < score
            for other_subset, other_score in score_by_subset.items()
            if set(other_subset).issubset(subset) and set(other_subset) != set(subset)
        )
        selected.append({
            **row,
            'is_main_complex': is_main,
            'mip_partition': (),
        })
    selected.sort(
        key=lambda item: (-float(item['synergy']), -len(item['subset_indices']), tuple(item['subset_indices']))
    )
    return selected


synergy_complexes = select_synergy_complexes(synergy_ranking)
print('按 EI 协同排序后的第一名:', subset_label(synergy_ranking[0]['subset_indices']), f"Syn_EI={synergy_ranking[0]['synergy']:.6f}")
print('EI 协同类比筛选得到的 complex 数量:', len(synergy_complexes))




按 EI 协同排序后的第一名: {A, B, C, D, E} Syn_EI=0.216917
EI 协同类比筛选得到的 complex 数量: 4


## 6. 把 EI 协同排序结果也列出来

这里有两个对照要看：

- 一是“按 EI 协同分数直接排序”时，谁排第一；
- 二是“如果照搬 complex 的包含关系筛选规则，但把打分从 $\Phi$ 换成 EI 协同”，最后会选出哪些子系统。


In [7]:
synergy_display_rows = []
for row in synergy_complexes:
    synergy_display_rows.append({
        'subset_indices': row['subset_indices'],
        'synergy': float(row['synergy']),
        'mip_partition': (),
        'main_complex': 'yes' if row['is_main_complex'] else 'no',
    })

render_rank_table(
    synergy_display_rows,
    title='EI-synergy complexes sorted by Syn_EI',
    score_key='synergy',
    extra_cols=[('main complex', 'main_complex')],
)



rank,subset,synergy,MIP,main complex
1,"{A, B, C, D, E}",0.216917,-,no
2,"{A, B, C, D}",0.216917,-,no
3,"{A, B, C, E}",0.216917,-,no
4,"{A, B, C}",0.216917,-,yes


## 7. 直接比较 $\Phi$ 排序和 EI 协同排序

下表把两个分数放在同一张表里。这里最重要的不是看小数点，而是看排序与筛选结论是否一致。


In [8]:
phi_by_subset = {tuple(row['subset_indices']): float(row['phi']) for row in phi_ranking}
synergy_by_subset = {tuple(row['subset_indices']): float(row['synergy']) for row in synergy_ranking}
phi_complex_set = {tuple(row['subset_indices']) for row in phi_complexes}
synergy_complex_set = {tuple(row['subset_indices']) for row in synergy_complexes}

comparison_rows = []
for subset in sorted(phi_by_subset.keys(), key=lambda item: (-phi_by_subset[item], -len(item), item)):
    comparison_rows.append({
        'subset': subset_label(subset),
        'phi': phi_by_subset[subset],
        'synergy': synergy_by_subset[subset],
        'iit_complex': 'yes' if subset in phi_complex_set else 'no',
        'ei_complex': 'yes' if subset in synergy_complex_set else 'no',
    })

html_parts = [
    "<table style='border-collapse:collapse;font-size:13px;'>",
    "<thead><tr><th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>subset</th>"
    "<th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>Phi</th>"
    "<th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>Syn_EI</th>"
    "<th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>IIT complex?</th>"
    "<th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>EI-synergy complex?</th></tr></thead><tbody>",
]
for row in comparison_rows[:15]:
    html_parts.append(
        '<tr>'
        f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{row['subset']}</td>"
        f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{row['phi']:.6f}</td>"
        f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{row['synergy']:.6f}</td>"
        f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{row['iit_complex']}</td>"
        f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{row['ei_complex']}</td>"
        '</tr>'
    )
html_parts.append('</tbody></table>')
display(HTML(''.join(html_parts)))



subset,Phi,Syn_EI,IIT complex?,EI-synergy complex?
"{A, B, C}",1.000000,0.216917,yes,yes
"{A, B, D}",0.500000,0.061278,yes,no
"{A, B, E}",0.500000,0.061278,yes,no
"{A, B}",0.078072,0.012483,no,no
"{A, C}",0.078072,0.012483,no,no
"{B, C}",0.078072,0.012483,no,no
"{A, D}",0.029049,0.000000,no,no
"{A, E}",0.029049,0.000000,no,no
"{B, D}",0.029049,0.000000,no,no
"{B, E}",0.029049,0.000000,no,no


## 8. 这个例子里，EI 协同选 complex 会与 IIT 2.0 一致吗？

结论是：**不完全一致。**

这个例子里会出现三个层面的差异：

1. **按 $\Phi$ 排序时，第一名是核心 `{A,B,C}`。**
   这是因为它确实有一个正的、由 MIP 定义出来的不可约整合量。

2. **按 EI 协同排序时，`{A,B,C}` 不再唯一领先。**
   在当前构造里，`{A,B,C}`、`{A,B,C,D}`、`{A,B,C,E}`、`{A,B,C,D,E}` 的协同分数会打平；如果按“分数优先、节点数次优先”的排序规则，整个 5 节点系统会排在最前面。

3. **因此，如果把“complex 的筛选规则”照搬到 EI 协同上，得到的结果会比 IIT 2.0 更松。**
   IIT 2.0 会把 `{A,B,C}` 识别成最强的 main complex，而 EI 协同的类比规则会同时保留更大的若干超集，因为它们并没有拿到更低的协同分数。

也就是说，在这个离散布尔例子里，EI 分解的协同信息和 $\Phi$ 都能看出核心 `{A,B,C}` 很重要，但**如果直接用 EI 协同来“选 complex”，它并不会自动与 IIT 2.0 一致**。根本原因在于：

- $\Phi$ 依赖 MIP，强调的是“最弱划分之后还剩多少不可约整体性”；
- EI 协同强调的是“整体 EI 超过各部分求和多少”；
- 这两个量都和“联合性”有关，但不是同一个函数，因此排序与 complex 边界不必重合。


In [9]:
final_summary = [
    f"- IIT 2.0 排名第一的 complex: {subset_label(phi_complexes[0]['subset_indices'])}, Phi = {phi_complexes[0]['phi']:.6f}",
    f"- EI 协同排序第一的子系统: {subset_label(synergy_ranking[0]['subset_indices'])}, Syn_EI = {synergy_ranking[0]['synergy']:.6f}",
    f"- IIT complex 集合: {[subset_label(row['subset_indices']) for row in phi_complexes]}",
    f"- EI-synergy complex 集合: {[subset_label(row['subset_indices']) for row in synergy_complexes]}",
]
display(Markdown('\n'.join(final_summary)))



- IIT 2.0 排名第一的 complex: {A, B, C}, Phi = 1.000000
- EI 协同排序第一的子系统: {A, B, C, D, E}, Syn_EI = 0.216917
- IIT complex 集合: ['{A, B, C}', '{A, B, D}', '{A, B, E}']
- EI-synergy complex 集合: ['{A, B, C, D, E}', '{A, B, C, D}', '{A, B, C, E}', '{A, B, C}']